In [ ]:
# -------------------------------
# 1. Load processed EDA dataset
# -------------------------------

import pandas as pd

df_eda = pd.read_csv("../data/processed/df_eda.csv")
print("Loaded dataset shape:", df_eda.shape)
display(df_eda.head())

In [ ]:
# Define features and targets
# proto_features = [col for col in df_eda.columns if col.startswith("proto_")]
features = ["log_byte_ratio", "log_dur", "log_sbytes", "log_dbytes", "log_sbytes_per_dur", "log_dbytes_per_dur"] # + proto_features
X = df_eda[features]
y = df_eda["Label"]

print("Feature matrix shape:", X.shape)
print("Target vector shape:", y.shape)

In [ ]:
# -------------------------------
# 2. Fit isolation forest
# -------------------------------

from sklearn.ensemble import IsolationForest

contamination = df_eda["Label"].sum()/len(df_eda)
print("Contamination fraction:", contamination)
# Model initialisation
model = IsolationForest(
    n_estimators=100,
    contamination=contamination,
    random_state=42
)

# Model fit
model.fit(X)

In [ ]:
# -------------------------------
# 3. Generate predictions and anomaly scores
# -------------------------------

anomaly_scores = model.score_samples(X)

pred_labels = model.predict(X)
pred_labels_binary = (pred_labels == -1).astype(int)

df_eda["anomaly_score"] = anomaly_scores
df_eda["pred_anomaly"] = pred_labels_binary

df_eda[["Label", "anomaly_score", "pred_anomaly"]].head()

In [ ]:
# -------------------------------
# 4. Evaluate and visualise predictions
# -------------------------------

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Quick stats
num_anomalies = df_eda["pred_anomaly"].sum()
print(f"Number of predicted anomalies: {num_anomalies}")

# Confusion matrix
cm = confusion_matrix(df_eda["Label"], df_eda["pred_anomaly"])
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", xticklabels=["Normal", "Anomaly"], yticklabels=["Normal", "Anomaly"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix: Isolation Forest vs True Labels")
plt.show()

# Classification report
print(classification_report(df_eda["Label"], df_eda["pred_anomaly"]))

In [ ]:
# Visualise anomalies in feature space
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df_eda,
    x="log_byte_ratio",
    y="log_dur",
    hue="Label",
    style="pred_anomaly",
    alpha=0.6,
    palette={0: "blue", 1: "red"}
)
plt.xlabel("Log(Byte Ratio)")
plt.ylabel("Log(Duration)")
plt.title("Predicted Anomalies vs True Attack Labels")
plt.show()